# КИМ 3.1. Полносвязные сети (MLP) — эталонное решение

> **Это эталонное решение** для преподавателя. Студентам выдаётся ноутбук-задание
> [`kim-03-mlp.ipynb`](./kim-03-mlp.ipynb) без заполненных ячеек.
>
> Код ниже — один из возможных вариантов решения; не единственный и не обязательно
> оптимальный. Приводится для сверки и подготовки к защите.

---
## Часть А. Регрессия — California Housing

### 0. Импорт

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from tensorflow import keras
from tensorflow.keras import layers

### 1. Загрузка и подготовка (Z-нормализация ПО TRAIN)

In [ ]:
(tr_data, tr_targets), (te_data, te_targets) = \
    keras.datasets.california_housing.load_data(version='large')

# Z-нормализация признаков по train-статистикам
mean = tr_data.mean(axis=0)
std  = tr_data.std(axis=0)
x_train = (tr_data - mean) / std
x_test  = (te_data - mean) / std      # тот же mean/std!

# Масштабирование цели (для стабильности обучения)
scale_y = 100000.0
y_train = tr_targets / scale_y
y_test  = te_targets / scale_y

print(x_train.shape, x_test.shape)

**Ответ:** среднее и СКО для нормализации берутся **только по обучающей
выборке**, чтобы избежать утечки информации из теста в обучение. Если нормировать
test своими статистиками, мы «подсматриваем» в тестовые данные, и оценка качества
становится оптимистично завышенной — модель может переобучиться под конкретное
разбиение.

### 2. Архитектура MLP для регрессии

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(8,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1),   # без активации — это регрессия
])
model.summary()

### 3. Компиляция и обучение

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
history = model.fit(x_train, y_train, batch_size=32, epochs=100,
                    validation_split=0.2, verbose=0)

### 4. Кривые обучения

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['loss'], label='train')
ax[0].plot(history.history['val_loss'], label='val')
ax[0].set_title('Loss (MSE)'); ax[0].legend(); ax[0].grid(True)
ax[1].plot(history.history['mae'], label='train')
ax[1].plot(history.history['val_mae'], label='val')
ax[1].set_title('MAE'); ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

### 5. Оценка на тесте и перевод в доллары

In [ ]:
test_loss, test_mae_scaled = model.evaluate(x_test, y_test, verbose=0)
test_mae_dollars = test_mae_scaled * scale_y
print(f'Тестовое MAE: {test_mae_scaled:.4f} (в масштабе модели)')
print(f'Тестовое MAE: {test_mae_dollars:,.0f} долларов (примерно)')

---
## Часть Б. Классификация — Fashion-MNIST

### 6. Загрузка и подготовка

In [ ]:
from tensorflow.keras import utils

(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
x_train = x_train.reshape(60000, 784).astype('float32') / 255.0
x_test  = x_test.reshape(10000, 784).astype('float32') / 255.0
y_train_oh = utils.to_categorical(y_train, 10)
y_test_oh  = utils.to_categorical(y_test, 10)

### 7. Архитектура MLP

In [ ]:
model = keras.Sequential([
    layers.Dense(800, activation='relu', input_dim=784),
    layers.Dense(300, activation='relu'),
    layers.Dense(10, activation='softmax'),
])
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

### 8. Обучение и оценка

In [ ]:
history = model.fit(x_train, y_train_oh, batch_size=128, epochs=20,
                    validation_split=0.2, verbose=0)
test_loss, test_acc = model.evaluate(x_test, y_test_oh, verbose=0)
print(f'Тестовая точность MLP 800-300: {test_acc:.4f}')
# Ожидается ~0.89

### 9. Сравнение архитектур

In [ ]:
import time
archs = {
    '800-10':     [layers.Dense(800, activation='relu', input_dim=784),
                   layers.Dense(10, activation='softmax')],
    '800-300-10': [layers.Dense(800, activation='relu', input_dim=784),
                   layers.Dense(300, activation='relu'),
                   layers.Dense(10, activation='softmax')],
    '256-128-10': [layers.Dense(256, activation='relu', input_dim=784),
                   layers.Dense(128, activation='relu'),
                   layers.Dense(10, activation='softmax')],
}
results = {}
for name, layers_list in archs.items():
    m = keras.Sequential(layers_list)
    m.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    t0 = time.time()
    m.fit(x_train, y_train_oh, batch_size=128, epochs=15, validation_split=0.2, verbose=0)
    dt = time.time() - t0
    _, acc = m.evaluate(x_test, y_test_oh, verbose=0)
    results[name] = (acc, dt)
    print(f'{name:14s}  test_acc={acc:.4f}  время={dt:.1f}с')

**Вывод:** глубокая сеть `800-300-10` даёт ~88–89 %, узкая `256-128-10` —
~88 %. Увеличение слоёв повышает точность, но и время обучения, и риск
переобучения. Для Fashion-MNIST оптимально 2 скрытых слоя умеренной ширины.

---
## Часть В. Универсальная теорема аппроксимации

**UAT (Cybenko 1989, Hornik 1991):** односкрытный MLP с сигмоидной (или более
общо — нелинейной) активацией может аппроксимировать любую непрерывную функцию на
ограниченном множестве с любой заданной точностью при достаточном числе нейронов.

**Оговорка «существование ≠ находимость»:** теорема гарантирует, что подходящая
аппроксимирующая сеть *существует*, но не что её можно *найти* градиентным
обучением. На практике:
- требуется экспоненциально много нейронов для точной аппроксимации в худшем случае;
- градиентный спуск не гарантирует глобальный оптимум;
- глубокие сети аппроксимируют многие функции экспоненциально эффективнее мелких
  (exponential depth efficiency), поэтому на практике предпочитают глубину, а не
  ширину одного скрытого слоя.